This code extracts data from images.
Coordinates for the starting/ending location of each position is required.


In [1]:
import pandas as pd
import os
import cv2
import numpy as np
import json

import imutils
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

#Load index list
Year='1935'
Showa='10'


In [2]:
#Encoding Function
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)


In [3]:
### CLOVA FUNCTION ###
import requests
import uuid
import time
import json
import cv2
import base64

api_url = 'https://deelieyxuc.apigw.ntruss.com/custom/v1/1972/ebd01bcbf693d069817622e9839e20914143c7d0d8953eddee40e8b0af96c95b/general'
secret_key = 'S1NmVXpYZlJ0cGJ0ZEFRZXVlbkRkaHFReE9FcHNTQ0U='

def Clova(file_data):
    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(file_data).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json    

In [4]:
#Function for Cell
def GetCell(cropped):
    #Code for Adding Grid
        ##Right page
        img = cropped.copy()
        hh, ww = img.shape[:2]

        #Identify grid location
        ## convert to grayscale
        gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
        # threshold gray image
        thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)[1]

        ## count number of non-zero pixels in each column and row. 
        countCol = np.count_nonzero(thresh, axis=0)
        countRow = np.count_nonzero(thresh, axis=1)

        ###############
        ## Column lines
        ###############
        ### This finds the height of the smallest peak
        peaksCol, _ = find_peaks(countCol, distance=10)
        ### threshold count at Thres
        count_thresh = countCol.copy()
        count_thresh[peaksCol] = 255
        count_thresh[count_thresh!=255] = 0
        count_thresh = count_thresh.astype(np.uint8)

        ### get contours
        contoursCol = cv2.findContours(count_thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contoursCol = contoursCol[0] if len(contoursCol) == 2 else contoursCol[1]

        ### loop over contours and get bounding boxes and ycenter and draw horizontal line at ycenter
        result = cropped.copy()
        for cntr in contoursCol:
            x,y,w,h = cv2.boundingRect(cntr)
            ycenter = y
            cv2.line(result, (ycenter,0), (ycenter,hh), (255, 0, 0), 1)
        

        ###############
        ## Row lines
        ###############
        peaksRow, _ = find_peaks(countRow, distance=60)
        ### threshold count at Thres
        count_thresh = countRow.copy()
        count_thresh[peaksRow] = 255
        count_thresh[count_thresh!=255] = 0
        count_thresh = count_thresh.astype(np.uint8)

        ### get contours
        contoursRow = cv2.findContours(count_thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contoursRow = contoursRow[0] if len(contoursRow) == 2 else contoursRow[1]

        ### loop over contours and get bounding boxes and ycenter and draw horizontal line at ycenter
        for cntr in contoursRow:
            x,y,w,h = cv2.boundingRect(cntr)
            ycenter = y+h//2
            cv2.line(result, (0,ycenter), (hh,ycenter), (255, 0, 0), 1)
                
        return(peaksRow,peaksCol)

In [5]:
def Extract(Position,ImageNumber):
    path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\"+Year+"\\"+Dept+"\\"+Office+"\\"+Position+"\\"
    stream = open(path+"Image"+str(ImageNumber)+'.png', "rb")
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

    HH,WW=img.shape[:2]
    
    dfA = pd.DataFrame(columns = ['Name', 'Wage'])
    dfT = pd.DataFrame(columns = ['Name', 'Wage'])
    dfB = pd.DataFrame(columns = ['Name', 'Wage'])
    
    if Position=="Admin":
        cropped=img[0:HH//2,0:WW]
        MiddleLineList=GetCell(cropped)[0]
        res = list(map(abs, [d-HH//4 for d in MiddleLineList.tolist()]))
        minpos = res.index(min(res))

        MiddleLine=MiddleLineList[minpos]
        ColumnLine=GetCell(cropped)[1]
        
        ##Top Chunk ##
        path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
        HH,WW=cropped.shape[:2]
        for Line in ColumnLine.tolist():
            if Line==ColumnLine.tolist()[0]:
                #Wage
                Image=cropped[0:MiddleLine,0:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Wage='NA'
                else:
                    Wage=''.join([d['inferText'] for d in Json['fields']])

                #Name
                Image=cropped[MiddleLine:HH,0:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfT = dfT.append(df2, ignore_index = True)
            else:
                #Wage
                Image=cropped[0:MiddleLine,Line-15:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Wage='NA'
                else:
                    Wage=''.join([d['inferText'] for d in Json['fields']])

                #Name
                Image=cropped[MiddleLine:HH,Line-15:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfT = dfT.append(df2, ignore_index = True)

        ##Bottom Chunk##
        cropped=img[HH//2:HH,0:WW]
        HH,WW=cropped.shape[:2]
        MiddleLineList=GetCell(cropped)[0]
        res = list(map(abs, [d-HH//4 for d in MiddleLineList.tolist()]))
        minpos = res.index(min(res))
        MiddleLine=MiddleLineList[minpos]
        ColumnLine=GetCell(cropped)[1]
        
        path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
        for Line in ColumnLine.tolist():
            if Line<15:
                #Wage
                Image=cropped[0:MiddleLine,0:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Wage='NA'
                else:
                    Wage=''.join([d['inferText'] for d in Json['fields']])

                #Name
                Image=cropped[MiddleLine:HH,0:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfB = dfB.append(df2, ignore_index = True)
            else:
                #Wage
                Image=cropped[0:MiddleLine,Line-15:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Wage='NA'
                else:
                    Wage=''.join([d['inferText'] for d in Json['fields']])

                #Name
                Image=cropped[MiddleLine:HH,Line-15:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfB = dfB.append(df2, ignore_index = True)
        return pd.concat([dfT,dfB], ignore_index = True)
    
    else:
        cropped=img

        HH,WW=cropped.shape[:2]
        MiddleLineList=GetCell(cropped)[0]
        res = list(map(abs, [d-HH//2 for d in MiddleLineList.tolist()]))
        minpos = res.index(min(res))
        MiddleLine=MiddleLineList[minpos]
        ColumnLine=GetCell(cropped)[1].tolist()
        
        path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
        for Line in ColumnLine:
            if Line<15:
                #Wage
                Image=cropped[0:MiddleLine,0:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Wage='NA'
                else:
                    Wage=''.join([d['inferText'] for d in Json['fields']])

                #Name
                Image=cropped[MiddleLine:HH,0:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfA = dfA.append(df2, ignore_index = True)

            else:
                #Wage
                Image=cropped[0:MiddleLine,Line-15:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Wage='NA'
                else:
                    Wage=''.join([d['inferText'] for d in Json['fields']])

                #Name
                Image=cropped[MiddleLine:HH,Line-15:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])


                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfA = dfA.append(df2, ignore_index = True)
        return(dfA)

In [6]:
#Load Data Frame
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\"+str(Year)+"\\DataFrame_Position.json"
with open(path, 'r') as j:
     dta = json.loads(j.read())

df = pd.read_csv(r'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/Index/S'+str(Showa)+'.csv')
df=df.drop(df.columns[0], axis=1)

In [45]:
#Test code| Version 2#
#Show Working office#
n=1

#Extract key info of office
Row  = df.iloc[n]
print(Row)
###Collect Location information###
Dept=Row["Dept"]
Office=Row["Office"]
PositionList=list(dta[str(Year)][Dept][Office]['Positions'].keys())
print(PositionList)

for Position in PositionList:
    StrPage=int(dta[str(Year)][Dept][Office]['Positions'][Position]['Starting_Page'])
    EndPage=int(dta[str(Year)][Dept][Office]['Positions'][Position]['EndPage'])
    PageList=list(set([1,EndPage-StrPage+1]))
    print(Position)
    for ImageNumber in PageList:        
        print('Image Number is '+str(ImageNumber))
        #Download Image
        path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\"+Year+"\\"+Dept+"\\"+Office+"\\"+Position+"\\"
        try:
            stream = open(path+"Image"+str(ImageNumber)+'.png', "rb")
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
        except:
            print('Could not find image')
            continue

        HH,WW=img.shape[:2]

        DF=pd.DataFrame(columns = ['Name', 'Wage'])
        if Position=='Admin':
            croppedTop=img[0:HH//2,0:WW]
            cv2.imshow("Sample",croppedTop)
            cv2.waitKey(0)

            croppedBtm=img[HH//2:HH,0:WW]
            cv2.imshow("Sample",croppedBtm)
            cv2.waitKey(0)
            
            dta[str(Year)][Dept][Office]['Positions'][Position]['Data']=Extract(Position,ImageNumber)

        else:
            dta[str(Year)][Dept][Office]['Positions'][Position]['Data']=Extract(Position,ImageNumber)


Office      職員課
Dept      Admin
Year         10
Page          3
Name: 1, dtype: object
['Manager', 'Admin']
Manager
Image Number is 1


C:\Temp\ipykernel_35268\2101252250.py:183: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_35268\2101252250.py:183: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_35268\2101252250.py:211: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_35268\2101252250.py:211: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_35268\2101252250.py:211: FutureWarning: The frame.append method is deprecated and will be removed 

Admin
Image Number is 1


C:\Temp\ipykernel_35268\2101252250.py:52: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_35268\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_35268\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_35268\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_35268\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from 

Image Number is 2
Could not find image


C:\Temp\ipykernel_35268\2101252250.py:143: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfB = dfB.append(df2, ignore_index = True)


In [8]:
FailedList=[]
#Show Working office#
for n in range(1,len(df)):
    #Extract key info of office
    Row  = df.iloc[n]
    print(Row)
    ###Collect Location information###
    Dept=Row["Dept"]
    Office=Row["Office"]
    try:
        PositionList=list(dta[str(Year)][Dept][Office]['Positions'].keys())
    except:
        print('Failed showing list of positions at '+Dept+Office)
        FailedList.append((n,Dept,Office))
        continue
    print(PositionList)

    for Position in PositionList:
        StrPage=int(dta[str(Year)][Dept][Office]['Positions'][Position]['Starting_Page'])
        EndPage=int(dta[str(Year)][Dept][Office]['Positions'][Position]['EndPage'])
        PageList=list(set([1,EndPage-StrPage+1]))
        print(Position)
        for ImageNumber in PageList:        
            #Download Image
            path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\"+Year+"\\"+Dept+"\\"+Office+"\\"+Position+"\\"
            try:
                stream = open(path+"Image"+str(ImageNumber)+'.png', "rb")
                bytes = bytearray(stream.read())
                numpyarray = np.asarray(bytes, dtype=np.uint8)
                img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
            except:
                print('Could not find image at'+Dept+Office+Position)
                FailedList.append((n,Dept,Office,Position))
                continue

            HH,WW=img.shape[:2]

            DF=pd.DataFrame(columns = ['Name', 'Wage'])
            if Position=='Admin':
                croppedTop=img[0:HH//2,0:WW]
                croppedBtm=img[HH//2:HH,0:WW]
                try:
                    dta[str(Year)][Dept][Office]['Positions'][Position]['Data']=Extract(Position,ImageNumber)
                except:
                    continue

            else:
                cropped=img
                try:
                    dta[str(Year)][Dept][Office]['Positions'][Position]['Data']=Extract(Position,ImageNumber)
                except:
                    print('Failed at extracting data at '+Dept+Office+Position)
                    FailedList.append((n,Dept,Office,Position))
                    continue


Office      職員課
Dept      Admin
Year         10
Page          3
Name: 1, dtype: object
['Manager', 'Admin']
Manager


C:\Temp\ipykernel_31124\2101252250.py:183: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:183: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:211: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:211: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:211: FutureWarning: The frame.append method is deprecated and will be removed 

Admin


C:\Temp\ipykernel_31124\2101252250.py:52: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from 

Could not find image atAdmin職員課Admin
Office      文書課
Dept      Admin
Year         10
Page          4
Name: 2, dtype: object
['Admin', 'Manager']
Admin


...

C:\Temp\ipykernel_31124\2101252250.py:183: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:211: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)


Admin


C:\Temp\ipykernel_31124\2101252250.py:52: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from 

Could not find image at養育院巣鴨分院Admin
Office    安房分院
Dept       養育院
Year        10
Page       151
Name: 63, dtype: object
Failed showing list of positions at 養育院安房分院


C:\Temp\ipykernel_31124\2101252250.py:143: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfB = dfB.append(df2, ignore_index = True)


In [52]:
Frame=pd.DataFrame(columns=['Dept', 'Office', 'Positions','Position','Name','Wage'])
FailedList=[]
for n in range(0,len(df)):
    Row=df.iloc[n]
    Dept=Row['Dept']
    Office=Row['Office']
    try:
        PosiList=list(dta[Year][Dept][Office]['Positions'].keys())
    except:
        FailedList.append(list((Dept,Office)))
        continue
        
    for Posi in PosiList:
        try:
            NewFrame=dta[Year][Dept][Office]['Positions'][Posi]['Data']
            NewFrame['Dept']=Dept
            NewFrame['Office']=Office
            NewFrame['Positions']=Posi
            Frame=pd.concat([Frame,NewFrame])
        except:
            FailedList.append(list((n,Dept,Office,Posi)))
            continue
Frame.to_csv("C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\"+Year+"\\Final.csv")

In [54]:
FailedList

[[0, 'Admin', '秘書課', 'Admin'],
 [0, 'Admin', '秘書課', 'Outsource'],
 [5, '監査局', '区政課', 'Outsource'],
 [6, '監査局', '統計課', 'Manager'],
 [6, '監査局', '統計課', 'Engineer2'],
 [6, '監査局', '統計課', 'Admin'],
 [6, '監査局', '統計課', 'Outsource'],
 ['監査局', '都市計画課'],
 ['財務局', '収納課'],
 [12, '財務局', '地理課', 'Outsource'],
 [13, '財務局', '会計課', 'Guard'],
 [14, '産業局', '庶務課', 'Outsource'],
 [17, '産業局', '権度課', 'Admin'],
 ['教育局', '庶務課'],
 [19, '教育局', '学務課', 'Manager'],
 [19, '教育局', '学務課', 'Admin'],
 [19, '教育局', '学務課', 'Engineer2'],
 [20, '教育局', '社会教育課', 'Admin'],
 [20, '教育局', '社会教育課', 'Engineer2'],
 [21, '教育局', '体育課', 'Admin'],
 [22, '教育局', '視学課', 'Inspector'],
 [22, '教育局', '視学課', 'Outsource'],
 [25, '社会局', '福利課', 'Manager'],
 [25, '社会局', '福利課', 'Engineer1'],
 [25, '社会局', '福利課', 'Admin'],
 [25, '社会局', '福利課', 'Engineer2'],
 [25, '社会局', '福利課', 'Outsource'],
 [26, '社会局', '職業課', 'Engineer1'],
 [26, '社会局', '職業課', 'Engineer2'],
 [26, '社会局', '職業課', 'Outsource'],
 [28, '保健局', '衛生課', 'Medic'],
 ['保健局', '清掃課'],
 [30, '保健局', '公園課',

In [13]:
#Extract key info of office
n=5
Row  = df.iloc[n]
print(Row)
###Collect Location information###
Dept=Row["Dept"]
Office=Row["Office"]
PositionList=list(dta[str(Year)][Dept][Office]['Positions'].keys())
print(PositionList)

for Position in PositionList:
    StrPage=int(dta[str(Year)][Dept][Office]['Positions'][Position]['Starting_Page'])
    EndPage=int(dta[str(Year)][Dept][Office]['Positions'][Position]['EndPage'])
    PageList=list(set([1,EndPage-StrPage+1]))
    print(Position)
    for ImageNumber in PageList:        
        #Download Image
        path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\"+Year+"\\"+Dept+"\\"+Office+"\\"+Position+"\\"
        try:
            stream = open(path+"Image"+str(ImageNumber)+'.png', "rb")
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
        except:
            print('Could not find image at'+Dept+Office+Position)
            FailedList.append((n,Dept,Office,Position))
            continue

        HH,WW=img.shape[:2]

        DF=pd.DataFrame(columns = ['Name', 'Wage'])
        if Position=='Admin':
            croppedTop=img[0:HH//2,0:WW]
            croppedBtm=img[HH//2:HH,0:WW]
            Frame=Extract(Position,ImageNumber)
            display(Frame)
            dta[str(Year)][Dept][Office]['Positions'][Position]['Data']=Extract(Position,ImageNumber)

        else:
            cropped=img
            Frame=Extract(Position,ImageNumber)
            display(Frame)
            dta[str(Year)][Dept][Office]['Positions'][Position]['Data']=Extract(Position,ImageNumber)

Office    区政課
Dept      監査局
Year       10
Page        7
Name: 5, dtype: object
['Manager', 'Admin', 'Outsource']
Manager


C:\Temp\ipykernel_31124\2101252250.py:183: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:211: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:211: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:211: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:211: FutureWarning: The frame.append method is deprecated and will be removed 

,Name,Wage
0,"日期,富前一五二九",
1,行政掛長機村英,之上
2,人生活路王和原超,
3,局監察課勤務)訟原(推,(照
4,"""text ""＼、、",
5,汁,
6,照理掛長圖本梹台,+
7,AKB,
8,課長谷III昇,
9,,


C:\Temp\ipykernel_31124\2101252250.py:183: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:211: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:211: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:211: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:211: FutureWarning: The frame.append method is deprecated and will be removed 

Admin


C:\Temp\ipykernel_31124\2101252250.py:52: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from 

,Name,Wage
0,,
1,,
2,,
3,,
4,,
5,,
6,,
7,古賀野成水,月七五
8,會根光造,月入不吉
9,松木常藏,


C:\Temp\ipykernel_31124\2101252250.py:52: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_31124\2101252250.py:78: FutureWarning: The frame.append method is deprecated and will be removed from 

Could not find image at監査局区政課Admin
Outsource
Could not find image at監査局区政課Outsource


C:\Temp\ipykernel_31124\2101252250.py:143: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfB = dfB.append(df2, ignore_index = True)


In [14]:
dta[str(Year)][Dept][Office]['Positions']

{'Manager': {'XLocation': 365,
  'Starting_Page': 1,
  'EndLocation': 213.0,
  'EndPage': 1.0,
  'Data':            Name Wage
  0     日期,富前一五二九     
  1       行政掛長機村英   之上
  2      人生活路王和原超     
  3   局監察課勤務)訟原(推   (照
  4    "text "＼、、     
  5             汁     
  6      照理掛長圖本梹台    +
  7           AKB     
  8       課長谷III昇     
  9                   
  10                 泰},
 'Admin': {'XLocation': 213,
  'Starting_Page': 1,
  'EndLocation': 25.0,
  'EndPage': 2.0,
  'Data':         Name  Wage
  0                 
  1                 
  2                 
  3                 
  4                 
  5                 
  6                 
  7      古賀野成水   月七五
  8       會根光造  月入不吉
  9       松木常藏      
  10    1%正八崎部    六上
  11  世界最高村佐久治     不
  12      野床四郎    五下
  13                
  14        NA    NA
  15              NA
  16              NA
  17     古賀野成水    NA
  18      會根光造    NA
  19      松木常藏    NA
  20      おもにせ    NA
  21    150近八崎    NA
  22  財源長高村佐久治    NA
  23         ! 

In [10]:
FailRate=len(FailedList)/len(df)
if len(FailedList)/len(df)<0.2:
    print('Fantastic!! Success Rate is '+str(1-(len(FailedList)/len(df))))
else:
    print('残念...Success Rate is '+str(1-(len(FailedList)/len(df))))
DF=pd.read_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')
DF.loc[int(Year)-1934, 'Position'] = 1-FailRate
DF.to_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')

残念...Success Rate is -0.09375


In [17]:
#For Multi-row image
#Top Row
#Creart data storage file


#Crop 
HHH=cropped.shape[:2][0]
MiddleLineList=GetCell(cropped)[0]
res = list(map(abs, [d-HHH//2 for d in MiddleLineList.tolist()]))
minpos = res.index(min(res))
MiddleLine=MiddleLineList[minpos]

ColumnLine=GetCell(cropped)[1]
MiddleLine=MiddleLineList[minpos]

path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
for Line in ColumnLine.tolist():
    if Line==ColumnLine.tolist()[0]:
        #Wage
        Image=cropped[0:MiddleLine,0:Line]
        cv2.imwrite(path+"Temp.jpg",Image)
        
        with open(path+"Temp.jpg",'rb') as f:
            file_data = f.read()
        Json=Clova(file_data)
        if Json['inferResult']=='ERROR':
            Wage='NA'
        else:
            Wage=''.join([d['inferText'] for d in Json['fields']])

        #Name
        Image=cropped[MiddleLine:HH,0:Line]
        cv2.imwrite(path+"Temp.jpg",Image)
        with open(path+"Temp.jpg",'rb') as f:
            file_data = f.read()
        Json=Clova(file_data)
        if Json['inferResult']=='ERROR':
            Name='NA'
        else:
            Name=''.join([d['inferText'] for d in Json['fields']])

        #Add to DF
        df2 = {'Name': Name, 'Wage': Wage}
        df = df.append(df2, ignore_index = True)
    else:
        #Wage
        Image=cropped[0:MiddleLine,Line-15:Line]
        cv2.imwrite(path+"Temp.jpg",Image)
        with open(path+"Temp.jpg",'rb') as f:
            file_data = f.read()
        Json=Clova(file_data)
        if Json['inferResult']=='ERROR':
            Wage='NA'
        else:
            Wage=''.join([d['inferText'] for d in Json['fields']])

        #Name
        Image=cropped[MiddleLine:HH,Line-15:Line]
        cv2.imwrite(path+"Temp.jpg",Image)
        with open(path+"Temp.jpg",'rb') as f:
            file_data = f.read()
        Json=Clova(file_data)
        if Json['inferResult']=='ERROR':
            Name='NA'
        else:
            Name=''.join([d['inferText'] for d in Json['fields']])

        #Add to DF
        df2 = {'Name': Name, 'Wage': Wage}
        df = df.append(df2, ignore_index = True)
dfT=df

C:\Temp\ipykernel_29772\815608455.py:47: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(df2, ignore_index = True)
C:\Temp\ipykernel_29772\815608455.py:73: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(df2, ignore_index = True)
C:\Temp\ipykernel_29772\815608455.py:73: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(df2, ignore_index = True)
C:\Temp\ipykernel_29772\815608455.py:73: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(df2, ignore_index = True)
C:\Temp\ipykernel_29772\815608455.py:73: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a f

In [28]:
MiddleLineList

array([ 29, 148], dtype=int64)

In [82]:
json_object = json.dumps(dta, indent=4,
                        cls=NpEncoder)
save_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'
with open(save_path+"DataFrame.json", "w") as outfile:
    outfile.write(json_object)

In [7]:
dta

{'1935': {'Admin': {'秘書課': {'Starting_Page': 2,
    'Office_X1': 416,
    'Ending_Page': 3,
    'Office_X2': 475,
    'Page_Range': [2, 3],
    'Positions': {'Admin': {'XLocation': 282,
      'Starting_Page': 1,
      'EndLocation': 198.0,
      'EndPage': 1.0},
     'Outsource': {'XLocation': 198,
      'Starting_Page': 1,
      'EndLocation': 0.0,
      'EndPage': 2.0}}},
   '職員課': {'Starting_Page': 3,
    'Office_X1': 465,
    'Ending_Page': 4,
    'Office_X2': 513,
    'Page_Range': [3, 4],
    'Positions': {'Manager': {'XLocation': 443,
      'Starting_Page': 1,
      'EndLocation': 244.0,
      'EndPage': 1.0},
     'Admin': {'XLocation': 244,
      'Starting_Page': 1,
      'EndLocation': 0.0,
      'EndPage': 2.0}}},
   '文書課': {'Starting_Page': 4,
    'Office_X1': 503,
    'Ending_Page': 5,
    'Office_X2': 185,
    'Page_Range': [4, 5],
    'Positions': {'Admin': {'XLocation': 256,
      'Starting_Page': 1,
      'EndLocation': 38.0,
      'EndPage': 2.0},
     'Manager': {'XL